# 06 — Ablation Study & Research Conclusions
### Beyond Static Acoustic Features: Temporal Dynamics, Silence Trimming, and Speaker Generalization in RAVDESS

**Final synthesis notebook · reproducible from saved artifacts · no audio extraction, no model fitting.**

This notebook consolidates **observed** outputs from notebooks 03–05. It does not silently recompute or replace the existing experiment results, and it does not call the historically inspected test split a new blind test.

**Reading guide**: CV is the 3-fold *speaker-grouped* cross-validation over **training actors 01–12**; fixed validation uses **actors 13–18**; historical test uses **actors 19–24** (already inspected during an earlier notebook). The unit of evaluation is a recording; actor-disjoint folds avoid sharing speakers across training and validation.

## 1. Environment & strict artifact verification

The notebook loads only small CSV/JSON files from the existing single project directory. It fails explicitly if an expected artifact is missing or has an unexpected schema — no auto-fabricated results and no silent substitution. Plots and summary tables are saved under `results/06_research_synthesis/`.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown, Image

try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive')
except ImportError:
    pass  # Allows a local environment with PROJECT_DIR adjusted below.

PROJECT_DIR = Path('/content/drive/MyDrive/speech-emotion-recognition-project-d')
V2_FEATURE_DIR = PROJECT_DIR / 'data/processed/ravdess_v2_delta_rmsdiff_20261104'
OLD_ML = PROJECT_DIR / 'results/04_ml'
NEW_ML = PROJECT_DIR / 'results/04_ml_v2'
OLD_EVAL = PROJECT_DIR / 'results/05_evaluation'
NEW_EVAL = PROJECT_DIR / 'results/05_evaluation_v2'
OUT = PROJECT_DIR / 'results/06_research_synthesis'
FIG = OUT / 'figures'
TAB = OUT / 'tables'
for folder in (FIG, TAB): folder.mkdir(parents=True, exist_ok=True)

required_files = {
    'baseline_config': OLD_ML / 'metrics/frozen_config.json',
    'baseline_comparison': OLD_ML / 'metrics/all_model_comparison.csv',
    'baseline_test': OLD_EVAL / 'final_test_metrics.csv',
    'v2_summary': NEW_ML / 'metrics/all_conditions.csv',
    'v2_folds': NEW_ML / 'metrics/grouped_cv_fold_scores.csv',
    'v2_manifest': NEW_ML / 'metrics/exploratory_v2_manifest.json',
    'v2_per_emotion': NEW_EVAL / 'cv_selected_validation_per_emotion.csv',
    'trim_diagnostics': V2_FEATURE_DIR / 'trimming_comparison.csv',
}
missing = [f'{name}: {path}' for name, path in required_files.items() if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing files from notebook 03–05; run the corresponding export cells first:\n' + '\n'.join(missing))

config = json.loads(required_files['baseline_config'].read_text(encoding='utf-8'))
manifest = json.loads(required_files['v2_manifest'].read_text(encoding='utf-8'))
original = pd.read_csv(required_files['baseline_comparison'])
historical = pd.read_csv(required_files['baseline_test'])
summary = pd.read_csv(required_files['v2_summary'])
folds = pd.read_csv(required_files['v2_folds'])
per_emotion = pd.read_csv(required_files['v2_per_emotion'])
trim = pd.read_csv(required_files['trim_diagnostics'])

assert len(summary) == 24 and len(folds) == 72, 'Expected 24 conditions × 3 folds.'
assert summary[['mode','representation','model']].drop_duplicates().shape[0] == 24
assert set(summary['mode']) == {'trim30','trim50','no_trim'}
assert set(summary['representation']) == {'static','enriched'}
assert set(summary['n_features']) == {39,120}
assert len(trim) == 1440 and trim['relative_path'].is_unique
assert set(manifest['train_actors']) == set(range(1,13))
assert set(manifest['validation_actors']) == set(range(13,19))
assert set(manifest['historically_inspected_test_actors']) == set(range(19,25))
assert config['fast_mode'] is False, 'Old baseline was generated in FAST_MODE.'
assert len(historical) == 1
assert {'cv_macro_f1','validation_macro_f1','cv_gap','cv_sd'}.issubset(summary.columns)
assert {'cv_f1','train_fold_f1','fold'}.issubset(folds.columns)
assert np.isfinite(summary[['cv_macro_f1','validation_macro_f1','cv_gap','cv_sd']]).all().all()
assert np.isfinite(folds[['cv_f1','train_fold_f1']]).all().all()

EMOTIONS = ['neutral','calm','happy','sad','angry','fearful','disgust','surprised']
BLUE, PURPLE, LIGHT, DARK = '#5471A6', '#875BB7', '#D9CCEB', '#273454'
sns.set_theme(style='whitegrid', font_scale=.94)
plt.rcParams.update({'figure.dpi':115,'savefig.dpi':170,'axes.spines.top':False,'axes.spines.right':False})

def save(fig, name):
    fig.savefig(FIG / name, bbox_inches='tight', dpi=170)
    plt.show()

print('Verified 24 model/feature conditions, 72 group-fold results, 1440 trimming records.')
print('CSV/JSON-only synthesis: no audio processing or model fitting.')

## 2. Experimental questions and protocol

**RQ1 — Temporal representation:** Does adding 81 dynamic summary features (Delta/Delta-Delta MFCC and RMS dynamics) to the original 39 static acoustic features improve *speaker-grouped* CV Macro F1?

**RQ2 — Trimming sensitivity:** How do `no_trim`, `trim50`, and `trim30` affect performance at fixed representation and model family?

**RQ3 — Generalization:** How consistent are the grouped-CV, held-out validation and historical-test results? Is memorization of training speakers a plausible issue?

These are *exploratory*, not confirmatory, experiments. We use the same recording keys, actor protocol and fixed model hyperparameters across the V2 3 × 2 × 4 conditions, but this dataset has only 24 actors and three CV folds. Treat fold summaries as descriptive; do not claim statistical significance or causal isolation without further tests.

In [ ]:
protocol = pd.DataFrame([
    {'partition':'Training / GroupKFold','actors':'01–12','recordings':720,'purpose':'Compare candidate families and feature conditions'},
    {'partition':'Fixed validation','actors':'13–18','recordings':360,'purpose':'Descriptive check of transfer to other actors'},
    {'partition':'Historical test','actors':'19–24','recordings':360,'purpose':'Old frozen-model report only; not a new blind benchmark'},
])
display(protocol)
assert {int(x) for x in config['train_actors']} == set(range(1,13))
assert {int(x) for x in config['validation_actors']} == set(range(13,19))
assert {int(x) for x in config['test_actors']} == set(range(19,25))
protocol.to_csv(TAB/'experimental_protocol.csv',index=False)

## 3. Representation ablation: paired fold scores (H1)

Each `enriched − static` difference compares the **same model, preprocessing condition and speaker fold**. We inspect individual differences as well as aggregated means, rather than interpreting all folds as independent experimental replications.

In [ ]:
paired = folds.pivot(index=['mode','model','fold'], columns='representation', values='cv_f1')
assert paired.shape == (36,2) and not paired.isna().any().any()
paired = paired.reset_index()
paired['delta_f1'] = paired['enriched'] - paired['static']
effects = paired.groupby('mode')['delta_f1'].agg(['mean','std','min','max','count']).reindex(['trim30','trim50','no_trim'])
assert (effects['count'] == 12).all()
print('GroupKFold enriched-minus-static difference (model × fold pairs, descriptive):')
display(effects.round(4))
effects.to_csv(TAB/'temporal_feature_effects_by_trim.csv')
paired.to_csv(TAB/'temporal_feature_paired_folds.csv',index=False)

fig, axes = plt.subplots(1,2,figsize=(13,4.8))
sns.boxplot(data=paired,x='mode',y='delta_f1',order=['trim30','trim50','no_trim'],color=LIGHT,ax=axes[0])
sns.stripplot(data=paired,x='mode',y='delta_f1',order=['trim30','trim50','no_trim'],hue='model',alpha=.8,ax=axes[0])
axes[0].axhline(0,c=DARK,ls='--',lw=1)
axes[0].set(xlabel='Trimming',ylabel='Enriched − static CV Macro F1',title='Paired speaker-fold differences')
axes[0].legend(fontsize=7,title='Model')
mode_effect = effects.reset_index().rename(columns={'index':'mode'})
sns.barplot(data=mode_effect,x='mode',y='mean',order=['trim30','trim50','no_trim'],color=PURPLE,errorbar=None,ax=axes[1])
axes[1].axhline(0,c=DARK,ls='--',lw=1)
axes[1].set(xlabel='Trimming',ylabel='Mean paired-fold difference',title='Mean effect across four model families')
for k,v in enumerate(mode_effect.set_index('mode').loc[['trim30','trim50','no_trim'],'mean']):
    axes[1].text(k,v+(0.003 if v>=0 else -.008),f'{v:+.3f}',ha='center')
plt.tight_layout(); save(fig,'01_temporal_feature_ablation.png')

**Interpretation.** A positive average effect supports *some benefit of dynamic summaries on the observed grouped CV splits*. It does not establish improved performance for every emotion, model or new speaker; see negative fold differences and validation disagreement below.

## 4. Preprocessing ablation: silence trimming (H2)

`top_db=30` is **more aggressive** than `top_db=50` for these recordings. Removed duration and classification quality are different outcome measures; plot them separately and compare model-by-model.

In [ ]:
removed_cols = ['removed_pct_trim30','removed_pct_trim50','removed_pct_no_trim']
assert set(removed_cols).issubset(trim.columns), 'Trimming comparison schema differs from notebook 03.'
removed = trim[removed_cols].agg(['mean','median','std']).T
removed.index = ['trim30','trim50','no_trim']
removed.index.name = 'mode'
removed.to_csv(TAB/'duration_removed_summary.csv')
display(removed.round(2))

fig,ax = plt.subplots(figsize=(8,4))
sns.barplot(data=removed.reset_index(),x='mode',y='mean',order=['trim30','trim50','no_trim'],color=BLUE,errorbar=None,ax=ax)
ax.set(title='Mean fraction of recording removed',xlabel='Preprocessing mode',ylabel='Duration removed (%)')
plt.tight_layout();save(fig,'02_trimmed_duration.png')

conditions = [f'{m} / {r}' for m in ['trim30','trim50','no_trim'] for r in ['static','enriched']]
for metric,filename,title in [
    ('cv_macro_f1','03_cv_ablation_heatmap.png','GroupKFold Macro F1 — primary exploratory comparison'),
    ('validation_macro_f1','04_validation_ablation_heatmap.png','Fixed validation Macro F1 — descriptive only')]:
    x=summary.assign(condition=lambda d:d['mode']+' / '+d['representation']).pivot(index='model',columns='condition',values=metric).reindex(columns=conditions)
    fig,ax=plt.subplots(figsize=(12,4.5))
    sns.heatmap(x,annot=True,fmt='.3f',cmap='Purples',linewidths=.8,linecolor='white',ax=ax,cbar_kws={'label':'Macro F1'})
    ax.set(title=title,xlabel='',ylabel='');ax.tick_params(axis='x',rotation=30)
    plt.tight_layout();save(fig,filename)

**Interpretation.** Check for interactions: whether the best trim mode depends on model family and representation. A higher preservation fraction does not imply improved emotion recognition; silence can provide context and can also dilute speech information.

## 5. Speaker generalization and candidate-selection discrepancy (H3)

The V2 *exploratory* candidate was chosen by the **training GroupKFold mean**. The highest fixed-validation score belongs to a different configuration. Report the disagreement without using validation (or the historical test) to retroactively rename the CV winner.

In [ ]:
best_cv = summary.sort_values(['cv_macro_f1','cv_sd','model','mode','representation'],ascending=[False,True,True,True,True]).iloc[0]
best_val = summary.sort_values('validation_macro_f1',ascending=False).iloc[0]
chosen=manifest['chosen']
assert (str(best_cv['model']),str(best_cv['mode']),str(best_cv['representation'])) == (chosen['model'],chosen['mode'],chosen['representation'])
np.testing.assert_allclose([best_cv.cv_macro_f1],[chosen['cv_macro_f1']],rtol=0,atol=1e-12)
leaders = pd.DataFrame([
    {'selection_basis':'Best grouped CV (V2 predeclared)','model':best_cv['model'],'mode':best_cv['mode'],'representation':best_cv['representation'],'cv_macro_f1':best_cv.cv_macro_f1,'validation_macro_f1':best_cv.validation_macro_f1},
    {'selection_basis':'Highest fixed validation (descriptive)','model':best_val['model'],'mode':best_val['mode'],'representation':best_val['representation'],'cv_macro_f1':best_val.cv_macro_f1,'validation_macro_f1':best_val.validation_macro_f1},
])
display(leaders.round(3))
leaders.to_csv(TAB/'selection_discrepancy.csv',index=False)

view=summary.copy()
view['condition']=view['mode']+' / '+view['representation']
fig,axes=plt.subplots(1,2,figsize=(14,5))
sns.scatterplot(data=view,x='cv_macro_f1',y='validation_macro_f1',hue='model',style='representation',s=85,ax=axes[0])
axes[0].scatter([best_cv.cv_macro_f1],[best_cv.validation_macro_f1],s=300,facecolors='none',edgecolors='black',linewidths=1.6)
axes[0].set(title='Grouped CV vs fixed validation',xlabel='Grouped CV Macro F1',ylabel='Fixed validation Macro F1')
axes[0].legend(fontsize=7,loc='best')
sns.scatterplot(data=view,x='cv_macro_f1',y='cv_gap',hue='representation',style='model',s=85,ax=axes[1],palette={'static':BLUE,'enriched':PURPLE})
axes[1].set(title='Train–CV generalization gap',xlabel='Grouped CV Macro F1',ylabel='Mean train − grouped CV Macro F1')
plt.tight_layout();save(fig,'05_generalization_diagnostics.png')

print('CV-selected V2 model:',best_cv['model'],best_cv['mode'],best_cv['representation'])
print('CV F1 / fixed validation F1:',round(best_cv.cv_macro_f1,3),round(best_cv.validation_macro_f1,3))
print('Validation leader (DESCRIPTIVE, not selection basis):',best_val['model'],best_val['mode'],best_val['representation'])
print('Validation leader F1:',round(best_val.validation_macro_f1,3))

## 6. What happened to each emotion? (V2 CV-selected model, validation only)

The classification metrics below were exported by Notebook 05 for **validation actors 13–18**. They must not be labeled as new test results.

In [ ]:
assert set(per_emotion['emotion'])==set(EMOTIONS)
assert {'precision','recall','f1','support'}.issubset(per_emotion.columns)
per_emotion=per_emotion.set_index('emotion').loc[EMOTIONS].reset_index()
display(per_emotion.round(3))
fig,ax=plt.subplots(figsize=(9,4.6))
sns.barplot(data=per_emotion.sort_values('f1'),x='f1',y='emotion',color=PURPLE,ax=ax)
ax.set(xlim=(0,1),xlabel='F1 on fixed validation',ylabel='',title='CV-selected V2 model: emotion-wise F1')
plt.tight_layout();save(fig,'06_v2_validation_emotions.png')

## 7. Historical baseline vs V2 — keep evaluation boundaries explicit

The old model's *historical* test score comes from the frozen baseline notebook. The V2 model was **not** evaluated on this test in the current work (`RUN_V2_HISTORICAL_TEST=False`). Therefore this section deliberately does **not** fabricate a V2 test score or claim a head-to-head test win.

In [ ]:
old_val=float(config['selected_validation_f1'])
old_test=float(historical['test_macro_f1'].iloc[0])
comparison=pd.DataFrame([
    {'system':'Old frozen selected pipeline','evaluation':'Validation (original selection)','macro_f1':old_val,'status':'Validation used in old selection'},
    {'system':'Old frozen selected pipeline','evaluation':'Historical test (previously inspected)','macro_f1':old_test,'status':'Not a fresh blind test'},
    {'system':'V2 grouped-CV leader','evaluation':'Training-actor grouped CV','macro_f1':float(best_cv.cv_macro_f1),'status':'Exploratory 3-fold CV'},
    {'system':'V2 grouped-CV leader','evaluation':'Fixed validation','macro_f1':float(best_cv.validation_macro_f1),'status':'Descriptive transfer check'},
])
display(comparison.round(3));comparison.to_csv(TAB/'evaluation_boundaries.csv',index=False)
print('Do not compare the historical test score numerically to V2 validation as if it were the same partition.')

## 8. Research journey: retain informative unsuccessful experiments

The earlier notebooks contain important diagnostics that should appear in the research narrative even where their scores did not improve. This is a **gallery of saved figures**, not a rerun of PCA, K-Means, tuning, learning curves or permutation importance. Missing optional figures will be reported, never silently substituted.

In [ ]:
from IPython.display import Image
journey = [
    ('PCA projections — overlapping acoustic classes',OLD_ML/'figures/pca_train.png'),
    ('PCA cumulative explained variance',OLD_ML/'figures/pca_explained_variance.png'),
    ('Unsupervised clustering diagnostics',OLD_ML/'figures/clustering_train.png'),
    ('Original six-model comparison',OLD_ML/'figures/model_comparison.png'),
    ('Static feature-group ablation',OLD_ML/'figures/ablation_heatmap.png'),
    ('Regularization versus validation',OLD_ML/'figures/regularization_validation.png'),
    ('Interpretability: permutation importance',OLD_EVAL/'figures/permutation_importance.png'),
    ('Historical-test confusion',OLD_EVAL/'figures/test_confusion_matrix.png'),
    ('Error differences by actor',OLD_EVAL/'figures/per_actor.png'),
    ('V2 CV-selected validation diagnostics',NEW_EVAL/'figures/cv_selected_validation_diagnostics.png'),
]
for title,path in journey:
    display(Markdown('**'+title+'**'))
    if path.is_file():
        display(Image(filename=str(path),width=740))
    else:
        print('Optional figure not found:',path)
print('Figures above are reused from earlier experiments; no training was performed.')

## 9. Reproducibility & limitations checklist

**Supported by these experiments:** (i) dynamic-summary features have a positive *mean paired grouped-CV* effect in each preprocessing mode; (ii) trimming makes a substantial difference to signal duration and its relationship to classification depends on the model; (iii) the grouped-CV and fixed-validation rankings disagree.

**Not established:** statistical significance; superiority for every unseen speaker; a new independent test improvement; causal evidence that temporal summaries alone explain performance; equivalence to end-to-end sequence modeling.

**Limitations:** one acted-English corpus; only 24 speakers; 3 CV folds; static and derivative statistics still discard detailed temporal ordering; only 4 fixed V2 model families; multiple exploratory comparisons; earlier exposure to historical test. Speaker-aware folds prevent cross-speaker overlap within each split but do not by themselves eliminate every possible preprocessing/leakage issue in a larger deployment.

**Historical negatives are results too:** overlapping PCA classes, weak unsupervised clustering and mixed regularization outcomes motivate the acoustic-representation and speaker-transfer experiments rather than being hidden.

In [ ]:
checks = pd.DataFrame([
    {'check':'Fixed 720/360/360 speaker partitions','passed':True,'evidence':'Original frozen configuration + V2 manifest'},
    {'check':'Six feature tables × four fixed model families','passed':len(summary)==24,'evidence':'all_conditions.csv'},
    {'check':'Three grouped folds per condition','passed':len(folds)==72,'evidence':'grouped_cv_fold_scores.csv'},
    {'check':'Static vs enriched means compared on identical folds','passed':len(paired)==36,'evidence':'paired-fold pivot'},
    {'check':'No V2 historical test in this synthesis','passed':True,'evidence':'No fitting/prediction functions called'},
    {'check':'V2 selected by grouped CV, not by test','passed':all([best_cv.model==chosen['model'],best_cv['mode']==chosen['mode'],best_cv.representation==chosen['representation']]),'evidence':'exploratory_v2_manifest.json'},
])
assert checks['passed'].all(), 'Reproducibility check failed.'
display(checks); checks.to_csv(TAB/'audit_checklist.csv',index=False)

## 10. Publication-ready summary generated from actual metrics

Use these automatically generated sentences as the starting point for the report. Interpretations are cautious and based on observed saved results, not on unrun benchmarks or significance tests.

In [ ]:
effects_map=effects['mean'].to_dict()
report = f"""# Research conclusions — empirical SER mini-project

**Research setting.** We compared 39 static acoustic features with 120 static-and-dynamic summary features under three silence-trimming settings and four fixed classical classifiers. We evaluated models using three speaker-grouped training CV folds (actors 01–12), with separate fixed validation actors (13–18). Historical test actors (19–24) had already been inspected in the original study.

**H1: Temporal feature summaries.** Mean paired grouped-CV changes (enriched minus static; aggregated across matching model–fold comparisons) were {effects_map['trim30']:+.3f} (`trim30`), {effects_map['trim50']:+.3f} (`trim50`) and {effects_map['no_trim']:+.3f} (`no_trim`) Macro F1. The positive means indicate a useful exploratory signal, with heterogeneous individual fold effects.

**H2: Trimming sensitivity.** Mean removed durations were {removed.loc['trim30','mean']:.2f}% (`trim30`) versus {removed.loc['trim50','mean']:.2f}% (`trim50`) and {removed.loc['no_trim','mean']:.2f}% (`no_trim`). Different retention of context did not produce a universal monotonic improvement across models.

**H3: New-speaker transfer.** The CV-selected V2 candidate ({best_cv['model']}; {best_cv['mode']}; {best_cv['representation']}) achieved grouped-CV Macro F1 {best_cv.cv_macro_f1:.3f} but fixed validation Macro F1 {best_cv.validation_macro_f1:.3f}, with mean train–CV gap {best_cv.cv_gap:.3f}. The descriptive validation leader ({best_val['model']}; {best_val['mode']}; {best_val['representation']}) achieved {best_val.validation_macro_f1:.3f} validation Macro F1 but was not the grouped-CV winner. Thus candidate rankings depend on which previously unseen speaker group is evaluated.

**Historical context.** The originally frozen selected model achieved Macro F1 {old_val:.3f} on its model-selection validation split and {old_test:.3f} on the historically inspected test split. These values are *not* evidence of a fresh blind V2 comparison. The V2 historical test was intentionally not run in Notebook 05.

**Conclusion.** Adding dynamic acoustic summaries improves average speaker-grouped CV in this study, but neither aggressive nor gentle silence trimming consistently dominates, and generalization to different actors remains unstable. We retain negative PCA, clustering and regularization experiments as part of the reproducible research path. Results should not be generalized beyond this small acted corpus without larger speaker-diverse external evaluation.
"""
(OUT/'research_conclusions.md').write_text(report,encoding='utf-8')
display(Markdown(report))
print('Saved research conclusions to:',OUT/'research_conclusions.md')